# 6주차 ② 과적합과 일반화 · 평가지표 — 실습 3~5

**목표**: 훈련 데이터를 일부러 줄여 **훈련 곡선과 검증 곡선이 갈라지는 순간**을 직접 만들고,
드롭아웃·배치 정규화를 적용해 비교하며, 혼동행렬로 **어디서 틀리는지**를 읽는다.

> **과제 제출 대상 노트북입니다.**

```
   [과소적합]              [적당]                  [과적합]
   공부를 덜 했다          원리를 이해했다          기출문제 답만 외웠다

   훈련 60% / 검증 58%     훈련 90% / 검증 88%      훈련 99% / 검증 82%  ★
      둘 다 낮다              둘 다 높다              훈련만 높다
```

> **핵심 메시지 ★ (출제 1순위)**: **과적합의 진단법은 한 줄입니다** —
> **훈련 손실은 계속 내려가는데 검증 손실이 올라가기 시작하는 지점.**
> 그 순간부터 모델은 **배우는 게 아니라 외우고** 있습니다.

| 분할 | 쓰임 | 몇 번 보나 |
|---|---|---|
| **훈련(train)** | 파라미터를 학습 | 매 epoch |
| **검증(validation)** | **설정을 고른다** (lr, 드롭아웃 비율, epoch 수) | 매 epoch |
| **테스트(test)** | **최종 성능 보고** | 딱 한 번 |

> 테스트로 설정을 고르면 **테스트에 과적합**됩니다. 그래서 **고르는 용도의 데이터를 따로** 둡니다.

## 실습 3 — 과적합 재현·관찰 ★★

```
   데이터가 적다  +  모델이 크다  +  오래 돌린다   →   외우기 시작한다
```

5주차 모델(파라미터 23만 개)에 **훈련 데이터를 1,000장으로 줄이면** 금방 외웁니다.

In [ ]:
# 셀 1 — 일부러 데이터를 줄인다
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
full  = datasets.FashionMNIST("data", train=True,  download=True, transform=tf)
test  = datasets.FashionMNIST("data", train=False, download=True, transform=tf)

SMALL = 1000                                   # ★ 훈련을 1,000장으로
small_train = Subset(full, range(SMALL))
val_set     = Subset(full, range(SMALL, SMALL + 2000))

small_loader = DataLoader(small_train, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_set,     batch_size=256, shuffle=False)
test_loader  = DataLoader(test,        batch_size=256, shuffle=False)
print("훈련", len(small_train), "장 / 검증", len(val_set), "장 / 테스트", len(test), "장")

In [ ]:
# 셀 2 — 훈련/검증 손실을 매 epoch 기록
def build(dropout=0.0, bn=False):
    torch.manual_seed(0)
    layers = [nn.Flatten(), nn.Linear(784, 256)]
    if bn: layers.append(nn.BatchNorm1d(256))          # ★ 선형 → BN → 활성함수
    layers += [nn.ReLU()]
    if dropout: layers.append(nn.Dropout(dropout))
    layers += [nn.Linear(256, 128)]
    if bn: layers.append(nn.BatchNorm1d(128))
    layers += [nn.ReLU()]
    if dropout: layers.append(nn.Dropout(dropout))
    layers += [nn.Linear(128, 10)]
    return nn.Sequential(*layers).to(device)

def run(model, epochs=30, lr=0.05):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    lossfn = nn.CrossEntropyLoss()
    tr_hist, va_hist = [], []
    for _ in range(epochs):
        model.train(); run_l = n = 0
        for xb, yb in small_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run_l += loss.item() * xb.size(0); n += xb.size(0)
        tr_hist.append(run_l / n)

        model.eval(); run_l = n = 0                      # ★ 검증
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                run_l += lossfn(model(xb), yb).item() * xb.size(0); n += xb.size(0)
        va_hist.append(run_l / n)
    return tr_hist, va_hist

tr, va = run(build())
plt.plot(tr, label="훈련 손실")
plt.plot(va, label="검증 손실")
plt.axvline(int(torch.tensor(va).argmin()), color="red", ls="--", label="검증 최저점")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("과적합 — 두 곡선이 갈라지는 순간"); plt.show()

print(f"검증 손실이 가장 낮았던 epoch : {int(torch.tensor(va).argmin())+1}")
print(f"마지막 epoch  | 훈련 {tr[-1]:.4f} / 검증 {va[-1]:.4f}")
print(f"검증 최저 시점 | 훈련 {tr[va.index(min(va))]:.4f} / 검증 {min(va):.4f}")

> **관찰 포인트 ★★**: **훈련 손실은 계속 0 쪽으로 내려갑니다.** 그런데 검증 손실은
> 어느 지점에서 **바닥을 찍고 다시 올라갑니다.** 빨간 점선이 그 지점입니다.
> **그 이후의 학습은 성능을 떨어뜨리고 있습니다.**

> **핵심 메시지 ★★**: 5주차에는 **테스트 정확도 하나**만 봤습니다.
> 그래서 *"더 오래 돌리면 더 좋아지겠지"* 라고 생각하기 쉬웠죠. **아닙니다.**
> **두 곡선을 함께 보는 것**이 오늘 배우는 가장 중요한 습관입니다.

> **조기 종료(early stopping)** 는 위 그림의 **빨간 점선에서 멈추는 것**입니다.
> 구현은 *"검증 손실이 N epoch 동안 안 좋아지면 중단"* 이 전부이고,
> 실무에서는 그 시점의 가중치를 `best_model.pt` 로 저장해 둡니다 (3교시 실습 7).
> ⚠️ **곡선이 안 갈라지면** `SMALL` 을 500 으로 줄여 보세요.

## 드롭아웃과 배치 정규화

```
   [훈련 시]  매 배치마다 무작위로 p 비율의 뉴런을 0 으로

      ○ ○ ○ ○ ○          ○ ✗ ○ ✗ ○        ✗ ○ ○ ○ ✗
      배치 1              배치 2            배치 3
      → 특정 뉴런에만 의존할 수 없게 된다  →  여러 경로로 풀도록 강제

   [추론 시]  전부 켠다. 대신 출력 크기를 맞춘다   ← model.eval() 이 이걸 한다 ★
```

> **핵심 메시지 ★★**: **5주차에 `model.eval()` 을 "일단 쓰라"고 한 이유가 이것입니다.**
> 드롭아웃은 **훈련할 때와 추론할 때 동작이 다릅니다.**

```
   5주차: 입력 데이터를 한 번 정규화했다  (transforms.Normalize)
   오늘:  각 층의 입력을 매 배치마다 정규화한다  (BatchNorm)

        Linear → BatchNorm → ReLU        ★ 이 순서
```

> **가중치 감쇠(weight decay)**: 큰 가중치에 벌점을 주어 모델을 단순하게 유지합니다.
> `optim.SGD(..., weight_decay=1e-4)` 한 줄이면 됩니다. 사용법만 알고 넘어갑니다.

## 실습 4 — 드롭아웃·BN 적용 후 재비교 ★

In [ ]:
# 셀 3 — 세 조건 비교
설정 = {
    "기본":              dict(dropout=0.0, bn=False),
    "드롭아웃 0.3":       dict(dropout=0.3, bn=False),
    "드롭아웃 + BN":      dict(dropout=0.3, bn=True),
}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, kw in 설정.items():
    tr, va = run(build(**kw))
    ax[0].plot(tr, label=name)
    ax[1].plot(va, label=name)
    print(f"{name:14s} | 훈련 {tr[-1]:.4f} | 검증 최저 {min(va):.4f} "
          f"(epoch {va.index(min(va))+1})")

ax[0].set_title("훈련 손실"); ax[1].set_title("검증 손실")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: 드롭아웃을 넣으면 **훈련 손실은 오히려 높아집니다.**
> 뉴런을 꺼 놓고 학습하니 당연합니다. 그런데 **검증 손실의 최저점이 낮아지거나 늦게 옵니다.**
> *"훈련 성적을 일부러 깎아서 실전 성적을 올린다"* — 이것이 일반화 기법의 본질입니다.

In [ ]:
# 셀 4 — eval() 을 빼먹으면 어떻게 되나 ★
model = build(dropout=0.3)
run(model, epochs=10)

def acc(m, use_eval):
    if use_eval: m.eval()
    else:        m.train()                # ★ 일부러 훈련 모드로 평가
    c = t = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (m(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

print(f"model.eval()  → {acc(model, True)*100:.2f}%")
print(f"model.eval()  → {acc(model, True)*100:.2f}%   ← 두 번 해도 같습니다")
print(f"model.train() → {acc(model, False)*100:.2f}%   ← 낮고, 실행할 때마다 달라진다")
print(f"model.train() → {acc(model, False)*100:.2f}%   ← 보세요, 값이 또 다릅니다")

> **결과 해석 ★★**: `train()` 모드로 평가하면 **정확도가 낮고, 두 번 실행하면 값이 다릅니다.**
> 드롭아웃이 매번 다른 뉴런을 끄기 때문입니다.
> **`model.eval()` 을 빼먹는 것은 학기 내내 가장 흔한 버그**입니다. 오늘 한 번 겪어 두세요.

> **출제 지점**: 드롭아웃과 배치 정규화 **둘 다** `train()`/`eval()` 로 동작이 갈립니다.
> 이것이 5주차부터 모드 전환 습관을 들인 이유입니다.

## 실습 5 — 혼동행렬·정밀도·재현율·F1

> 정의는 선행 과목(빅데이터분석프로그래밍 13·14주)에서 이미 배웠습니다.
> 여기서는 **PyTorch 결과를 sklearn 에 어떻게 넘기는가**와 **혼동행렬을 읽는 법**만 봅니다.

In [ ]:
# 셀 5 — 전체 예측 수집
LABELS = ["티셔츠","바지","풀오버","드레스","코트","샌들","셔츠","스니커즈","가방","앵클부츠"]

model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        all_pred.append(model(xb).argmax(1).cpu())      # ★ .cpu() 를 빼면 sklearn 에서 오류
        all_true.append(yb)
y_pred = torch.cat(all_pred).numpy()
y_true = torch.cat(all_true).numpy()
print("예측 수집 완료 :", y_pred.shape)

In [ ]:
# 셀 6 — 지표와 혼동행렬
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(LABELS, rotation=90)
ax.set_yticks(range(10)); ax.set_yticklabels(LABELS)
ax.set_xlabel("예측"); ax.set_ylabel("정답"); ax.set_title("혼동행렬")
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# 셀 7 — 대각선 밖에서 가장 큰 혼동 5쌍
import numpy as np

off = cm.copy()
np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:5]

print("가장 자주 헷갈리는 조합")
for i, j in pairs:
    print(f"  정답 {LABELS[i]:8s} → 예측 {LABELS[j]:8s} : {off[i, j]}건")

> **관찰 포인트 ★★**: 대각선이 정답입니다. **대각선 밖의 큰 숫자를 찾으세요.**
> **셔츠 ↔ 티셔츠 ↔ 코트 ↔ 풀오버** 구역이 진하게 나옵니다.
> 5주차 실습 9에서 이미지로 봤던 그것을 **숫자로 확인**하는 것입니다.

```
   예: 100건 중 양성 2건인 불량품 검사

     "전부 정상"이라고만 답하는 모델  →  정확도 98%  ★ 그런데 쓸모가 0
                                          재현율 0%
```

| 지표 | 언제 중요한가 |
|---|---|
| **정밀도(precision)** | 잘못된 경보의 비용이 클 때 (스팸함에 중요 메일이 가면 안 됨) |
| **재현율(recall)** | 놓치는 비용이 클 때 (암 진단에서 환자를 놓치면 안 됨) |
| **F1** | 둘의 균형 |

> **핵심 메시지 ★**: FashionMNIST 는 클래스가 고르게 있어서 정확도로 충분합니다.
> 그런데 **현실 데이터는 대개 불균형**합니다. *"정확도 98%"* 를 보면
> **먼저 클래스 비율을 확인하는 습관**을 가지세요. 중간고사 출제 지점입니다.

---

### 과제 (마감 10/15 목 23:59)

```
  ① 조건 4개 이상을 바꾼 비교표  ★ 핵심
       (옵티마이저 / 학습률 / 드롭아웃 비율 / 배치 크기 중 택)
  ② 훈련·검증 곡선 캡처
  ③ TensorBoard 화면 1장       (3교시 실습 9)
  ④ 가장 좋았던 설정과 그 이유 3줄
  ⑤ best_model.pt 커밋 여부 명시
  ⑥ 커밋 · push · LMS 제출
```

> **채점의 초점은 "정확도가 높은 사람"이 아닙니다.**
> ① **조건을 하나씩만 바꿔 비교가 성립하게 실험했는가**
> ② **결과를 자기 말로 해석했는가**

### 이 노트북 체크리스트

- [ ] 과적합의 진단법을 **한 줄로** 말할 수 있다 ★★
- [ ] 검증이 테스트와 별도로 필요한 이유를 안다
- [ ] 데이터를 1,000장으로 줄여 **두 곡선이 갈라지는 것**을 직접 봤다 ★
- [ ] 드롭아웃이 훈련 손실을 **높이는데도** 쓰는 이유를 안다
- [ ] `model.eval()` 을 빼면 정확도가 낮고 매번 달라지는 것을 확인했다 ★★
- [ ] BN 의 위치(선형 → BN → 활성함수)를 안다
- [ ] `classification_report` 와 혼동행렬을 출력했다
- [ ] 정확도만 보면 안 되는 경우를 예로 들 수 있다